In [1]:
import pandas as pd

print("Pandas imported successfully!")

Pandas imported successfully!


In [2]:
from pathlib import Path

project_root = Path.cwd()

if not (project_root / "data" / "raw").is_dir():
    project_root = project_root.parent

data_path = project_root / "data" / "raw"

datasets = {
    "plans": pd.read_csv(data_path / "plans.csv"),
    "customers": pd.read_csv(data_path / "customers.csv"),
    "subscriptions": pd.read_csv(data_path / "subscriptions.csv"),
    "subscription_months": pd.read_csv(data_path / "subscription_months.csv"),
    "invoices": pd.read_csv(data_path / "invoices.csv")
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

plans: 10 rows, 11 columns
customers: 1200 rows, 12 columns
subscriptions: 1381 rows, 12 columns
subscription_months: 19015 rows, 15 columns
invoices: 11257 rows, 17 columns


In [3]:
# Inspect column names, data types, and missing values

for name, df in datasets.items():

    print(f"\n{'=' * 65}")
    print(f"DATASET: {name.upper()}")
    print(f"{'=' * 65}")

    schema = pd.DataFrame({
        "Column_Name": df.columns,
        "Data_Type": df.dtypes.astype(str).values,
        "Non_Null_Count": df.notna().sum().values,
        "Missing_Count": df.isna().sum().values
    })

    print(schema.to_string(index=False))


DATASET: PLANS
         Column_Name Data_Type  Non_Null_Count  Missing_Count
             plan_id       str              10              0
           plan_name       str              10              0
                tier       str              10              0
    billing_interval       str              10              0
      list_price_usd     int64              10              0
   monthly_price_usd   float64              10              0
      billing_months     int64              10              0
      seats_included     int64              10              0
extra_seat_price_usd     int64              10              0
           is_active       str              10              0
       launched_date       str              10              0

DATASET: CUSTOMERS
        Column_Name Data_Type  Non_Null_Count  Missing_Count
        customer_id       str            1200              0
       company_name       str            1200              0
            country       str        

In [4]:
# Combine schema information from all five datasets

schema_tables = []

for name, df in datasets.items():

    schema = pd.DataFrame({
        "Dataset": name,
        "Column_Name": df.columns,
        "Data_Type": df.dtypes.astype(str).values,
        "Non_Null_Count": df.notna().sum().values,
        "Missing_Count": df.isna().sum().values
    })

    schema_tables.append(schema)

# Combine all schemas into one DataFrame
combined_schema = pd.concat(schema_tables, ignore_index=True)

# Define output file location
output_path = project_root / "docs" / "Dataset_Schema.csv"

# Export schema to CSV
combined_schema.to_csv(output_path, index=False)

# Verification
print("Schema saved successfully!")
print("Total columns documented:", len(combined_schema))
print("Saved to:", output_path)

Schema saved successfully!
Total columns documented: 67
Saved to: c:\Users\ANKIT TRIPATHI\Desktop\SaaS-Revenue-Subscription-Analytics\docs\Dataset_Schema.csv


In [5]:
# Step 3.6 Part A — Missing Values Analysis

for name, df in datasets.items():

    print(f"\n{'=' * 60}")
    print(f"DATASET: {name.upper()}")
    print(f"{'=' * 60}")

    missing_count = df.isna().sum()

    missing_percent = (
        missing_count / len(df) * 100
    ).round(2)

    missing_summary = pd.DataFrame({
        "Missing_Count": missing_count,
        "Missing_Percentage": missing_percent
    })

    print(missing_summary.to_string())

    print(
        f"\nTotal Missing Values: {missing_count.sum()}"
    )


DATASET: PLANS
                      Missing_Count  Missing_Percentage
plan_id                           0                 0.0
plan_name                         0                 0.0
tier                              0                 0.0
billing_interval                  0                 0.0
list_price_usd                    0                 0.0
monthly_price_usd                 0                 0.0
billing_months                    0                 0.0
seats_included                    0                 0.0
extra_seat_price_usd              0                 0.0
is_active                         0                 0.0
launched_date                     0                 0.0

Total Missing Values: 0

DATASET: CUSTOMERS
                     Missing_Count  Missing_Percentage
customer_id                      0                0.00
company_name                     0                0.00
country                          0                0.00
region                           0             

In [6]:
# Overall missing-value summary

total_missing = sum(
    df.isna().sum().sum()
    for df in datasets.values()
)

print("Total missing values across all datasets:", total_missing)

Total missing values across all datasets: 2019


In [7]:
# Step 3.6 Part B — Exact Duplicate Records

print("EXACT DUPLICATE ROW ANALYSIS")
print("=" * 55)

total_duplicates = 0

for name, df in datasets.items():

    duplicate_count = df.duplicated().sum()

    print(f"{name}: {duplicate_count} duplicate rows")

    total_duplicates += duplicate_count

print("=" * 55)
print("Total exact duplicate rows:", total_duplicates)

EXACT DUPLICATE ROW ANALYSIS
plans: 0 duplicate rows
customers: 0 duplicate rows
subscriptions: 0 duplicate rows
subscription_months: 0 duplicate rows
invoices: 0 duplicate rows
Total exact duplicate rows: 0


In [8]:
# Check uniqueness of primary key candidates

primary_keys = {
    "plans": "plan_id",
    "customers": "customer_id",
    "subscriptions": "subscription_id",
    "subscription_months": "record_id",
    "invoices": "invoice_id"
}

print("PRIMARY KEY DUPLICATE ANALYSIS")
print("=" * 60)

for name, key in primary_keys.items():

    df = datasets[name]

    duplicate_keys = df[key].duplicated().sum()
    missing_keys = df[key].isna().sum()

    print(
        f"{name} | Key: {key} | "
        f"Duplicate IDs: {duplicate_keys} | "
        f"Missing IDs: {missing_keys}"
    )

PRIMARY KEY DUPLICATE ANALYSIS
plans | Key: plan_id | Duplicate IDs: 0 | Missing IDs: 0
customers | Key: customer_id | Duplicate IDs: 0 | Missing IDs: 0
subscriptions | Key: subscription_id | Duplicate IDs: 0 | Missing IDs: 0
subscription_months | Key: record_id | Duplicate IDs: 0 | Missing IDs: 0
invoices | Key: invoice_id | Duplicate IDs: 0 | Missing IDs: 0


In [9]:
# Check duplicate customer-month and subscription-month combinations

monthly_df = datasets["subscription_months"]

customer_month_duplicates = monthly_df.duplicated(
    subset=["customer_id", "period_month"]
).sum()

subscription_month_duplicates = monthly_df.duplicated(
    subset=["subscription_id", "period_month"]
).sum()

print("MONTHLY BUSINESS-GRAIN VALIDATION")
print("=" * 55)

print(
    "Duplicate customer-month combinations:",
    customer_month_duplicates
)

print(
    "Duplicate subscription-month combinations:",
    subscription_month_duplicates
)

MONTHLY BUSINESS-GRAIN VALIDATION
Duplicate customer-month combinations: 0
Duplicate subscription-month combinations: 0


In [10]:
# Step 3.6 Part C — Foreign Key Validation

relationships = [
    ("subscriptions", "customer_id", "customers", "customer_id"),
    ("subscriptions", "plan_id", "plans", "plan_id"),

    ("subscription_months", "customer_id", "customers", "customer_id"),
    ("subscription_months", "subscription_id", "subscriptions", "subscription_id"),
    ("subscription_months", "plan_id", "plans", "plan_id"),

    ("invoices", "customer_id", "customers", "customer_id"),
    ("invoices", "subscription_id", "subscriptions", "subscription_id")
]

fk_results = []

for child_table, child_key, parent_table, parent_key in relationships:

    child_df = datasets[child_table]
    parent_df = datasets[parent_table]

    # Find child values missing from the parent table
    orphan_mask = (
        child_df[child_key].notna()
        & ~child_df[child_key].isin(parent_df[parent_key].dropna())
    )

    orphan_count = int(orphan_mask.sum())
    null_fk_count = int(child_df[child_key].isna().sum())

    fk_results.append({
        "Child_Table": child_table,
        "Foreign_Key": child_key,
        "Parent_Table": parent_table,
        "Orphan_Records": orphan_count,
        "Null_Foreign_Keys": null_fk_count
    })

fk_summary = pd.DataFrame(fk_results)

print("FOREIGN KEY VALIDATION RESULTS")
print("=" * 80)
print(fk_summary.to_string(index=False))

FOREIGN KEY VALIDATION RESULTS
        Child_Table     Foreign_Key  Parent_Table  Orphan_Records  Null_Foreign_Keys
      subscriptions     customer_id     customers               0                  0
      subscriptions         plan_id         plans               0                  0
subscription_months     customer_id     customers               0                  0
subscription_months subscription_id subscriptions               0                  0
subscription_months         plan_id         plans               0                  0
           invoices     customer_id     customers               0                  0
           invoices subscription_id subscriptions               0                  0


In [11]:
# Overall relationship validation

total_orphans = fk_summary["Orphan_Records"].sum()
total_null_fks = fk_summary["Null_Foreign_Keys"].sum()

print("Total relationships checked:", len(fk_summary))
print("Total orphan records:", total_orphans)
print("Total null foreign keys:", total_null_fks)

if total_orphans == 0 and total_null_fks == 0:
    print("RESULT: PASS - All foreign key checks passed.")
else:
    print("RESULT: REVIEW REQUIRED - Relationship issues found.")

Total relationships checked: 7
Total orphan records: 0
Total null foreign keys: 0
RESULT: PASS - All foreign key checks passed.


In [16]:
# Step 3.6 Part D — Corrected Date Format Validation

date_columns = {
    "plans": ["launched_date"],
    "customers": ["signup_date", "churn_date", "cohort_month"],
    "subscriptions": ["start_date", "end_date"],
    "subscription_months": [
        "period_month", "month_start_date", "cohort_month"
    ],
    "invoices": [
        "invoice_date", "invoice_month",
        "billing_period_start", "billing_period_end"
    ]
}

# Columns using DD-MM-YYYY
day_first_columns = {
    ("customers", "signup_date"),
    ("customers", "churn_date"),
    ("subscriptions", "start_date"),
    ("subscriptions", "end_date")
}

date_results = []
parsed_dates = {}

for table_name, columns in date_columns.items():

    df = datasets[table_name]
    parsed_dates[table_name] = {}

    for column in columns:

        raw = df[column]

        if (table_name, column) in day_first_columns:
            parsed = pd.to_datetime(
                raw,
                format="%d-%m-%Y",
                errors="coerce"
            )
        else:
            parsed = pd.to_datetime(
                raw,
                errors="coerce"
            )

        parsed_dates[table_name][column] = parsed

        invalid_count = int(
            (raw.notna() & parsed.isna()).sum()
        )

        date_results.append({
            "Dataset": table_name,
            "Column": column,
            "Missing_Values": int(raw.isna().sum()),
            "Invalid_Dates": invalid_count
        })

date_summary = pd.DataFrame(date_results)

print("CORRECTED DATE FORMAT VALIDATION")
print("=" * 75)
print(date_summary.to_string(index=False))

print("\nTotal invalid dates:",
      int(date_summary["Invalid_Dates"].sum()))

CORRECTED DATE FORMAT VALIDATION
            Dataset               Column  Missing_Values  Invalid_Dates
              plans        launched_date               0              0
          customers          signup_date               0              0
          customers           churn_date             855              0
          customers         cohort_month               0              0
      subscriptions           start_date               0              0
      subscriptions             end_date             855              0
subscription_months         period_month               0              0
subscription_months     month_start_date               0              0
subscription_months         cohort_month               0              0
           invoices         invoice_date               0              0
           invoices        invoice_month               0              0
           invoices billing_period_start               0              0
           invoices   billing_p

C:\Users\ANKIT TRIPATHI\AppData\Local\Temp\ipykernel_1284\287681157.py:43: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  parsed = pd.to_datetime(
C:\Users\ANKIT TRIPATHI\AppData\Local\Temp\ipykernel_1284\287681157.py:43: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  parsed = pd.to_datetime(


In [17]:
# Step 3.6 Part D.3 — Date Order Validation

date_order_checks = [
    ("customers", "signup_date", "churn_date"),
    ("subscriptions", "start_date", "end_date"),
    ("invoices", "billing_period_start", "billing_period_end")
]

date_order_results = []

for table, start_col, end_col in date_order_checks:

    start = parsed_dates[table][start_col]
    end = parsed_dates[table][end_col]

    # Only compare rows where both dates are available
    valid_pairs = start.notna() & end.notna()

    # Find records where end date is earlier than start date
    invalid_order = int(
        (valid_pairs & (end < start)).sum()
    )

    date_order_results.append({
        "Dataset": table,
        "Start_Column": start_col,
        "End_Column": end_col,
        "Records_Checked": int(valid_pairs.sum()),
        "Invalid_Date_Order": invalid_order
    })

date_order_summary = pd.DataFrame(date_order_results)

print("DATE ORDER VALIDATION")
print("=" * 85)
print(date_order_summary.to_string(index=False))

print("\nTotal invalid date-order records:",
      int(date_order_summary["Invalid_Date_Order"].sum()))

DATE ORDER VALIDATION
      Dataset         Start_Column         End_Column  Records_Checked  Invalid_Date_Order
    customers          signup_date         churn_date              345                   0
subscriptions           start_date           end_date              526                   0
     invoices billing_period_start billing_period_end            11257                   0

Total invalid date-order records: 0


In [14]:
# Inspect original date formats without changing data

date_samples = [
    ("customers", "signup_date"),
    ("customers", "churn_date"),
    ("subscriptions", "start_date"),
    ("subscriptions", "end_date")
]

for table, column in date_samples:
    print(f"\n{table}.{column}")

    print(
        datasets[table][column]
        .dropna()
        .astype(str)
        .head(10)
        .to_list()
    )


customers.signup_date
['07-09-2021', '16-05-2023', '05-09-2022', '12-10-2021', '30-05-2021', '11-11-2023', '08-01-2023', '08-03-2021', '02-04-2021', '11-02-2021']

customers.churn_date
['01-04-2024', '01-11-2025', '01-01-2025', '01-04-2024', '01-02-2026', '01-03-2026', '01-07-2024', '01-10-2025', '01-03-2024', '01-10-2025']

subscriptions.start_date
['07-09-2021', '16-05-2023', '01-08-2024', '01-10-2024', '05-09-2022', '12-10-2021', '01-03-2025', '30-05-2021', '04-09-2024', '11-11-2023']

subscriptions.end_date
['01-08-2024', '01-10-2024', '01-03-2025', '29-02-2024', '31-03-2024', '01-10-2025', '31-10-2025', '31-12-2024', '01-06-2024', '31-03-2024']


In [18]:
# Step 3.6 Part D.4 — Monthly Reporting Date Validation

monthly = parsed_dates["subscription_months"]

period = monthly["period_month"]
month_start = monthly["month_start_date"]

# Check 1: Missing dates
missing_period = int(period.isna().sum())
missing_month_start = int(month_start.isna().sum())

# Check 2: Both columns represent the same month
valid_pairs = period.notna() & month_start.notna()

month_mismatches = int(
    (
        valid_pairs
        & (period.dt.to_period("M") != month_start.dt.to_period("M"))
    ).sum()
)

# Check 3: Month start date must be the first day
not_first_day = int(
    (
        month_start.notna()
        & (month_start.dt.day != 1)
    ).sum()
)

print("MONTHLY REPORTING DATE VALIDATION")
print("=" * 55)

print("Total monthly records:", len(period))
print("Missing period_month:", missing_period)
print("Missing month_start_date:", missing_month_start)
print("Period-month mismatches:", month_mismatches)
print("Month-start dates not on day 1:", not_first_day)

if (
    missing_period == 0
    and missing_month_start == 0
    and month_mismatches == 0
    and not_first_day == 0
):
    print("\nRESULT: PASS - Monthly reporting dates are consistent.")
else:
    print("\nRESULT: REVIEW REQUIRED - Date issues detected.")

MONTHLY REPORTING DATE VALIDATION
Total monthly records: 19015
Missing period_month: 0
Missing month_start_date: 0
Period-month mismatches: 17189
Month-start dates not on day 1: 17189

RESULT: REVIEW REQUIRED - Date issues detected.


In [19]:
# Step 3.6 Part E.1 — Numeric Data Quality Audit

numeric_results = []

for table_name, df in datasets.items():

    numeric_df = df.select_dtypes(include="number")

    for column in numeric_df.columns:

        values = numeric_df[column]

        numeric_results.append({
            "Dataset": table_name,
            "Column": column,
            "Data_Type": str(values.dtype),
            "Missing_Count": int(values.isna().sum()),
            "Negative_Count": int((values < 0).sum()),
            "Zero_Count": int((values == 0).sum()),
            "Min_Value": values.min(),
            "Max_Value": values.max()
        })

numeric_summary = pd.DataFrame(numeric_results)

print("NUMERIC DATA QUALITY AUDIT")
print("=" * 90)

print(numeric_summary.to_string(index=False))

print("\nTotal numeric columns:", len(numeric_summary))

print(
    "Total negative values:",
    int(numeric_summary["Negative_Count"].sum())
)

print(
    "Total zero values:",
    int(numeric_summary["Zero_Count"].sum())
)

NUMERIC DATA QUALITY AUDIT
            Dataset               Column Data_Type  Missing_Count  Negative_Count  Zero_Count  Min_Value  Max_Value
              plans       list_price_usd     int64              0               0           0      39.00   14990.00
              plans    monthly_price_usd   float64              0               0           0      39.00    1499.00
              plans       billing_months     int64              0               0           0       1.00      12.00
              plans       seats_included     int64              0               0           0       2.00     100.00
              plans extra_seat_price_usd     int64              0               0           0      10.00      25.00
      subscriptions                seats     int64              0               0           0       1.00     434.00
      subscriptions         discount_pct     int64              0               0         961       0.00      20.00
      subscriptions              mrr_usd   fl

In [20]:
# Identify columns containing negative values

negative_columns = numeric_summary[
    numeric_summary["Negative_Count"] > 0
]

print("COLUMNS WITH NEGATIVE VALUES")
print("=" * 75)

if negative_columns.empty:
    print("No negative numeric values found.")
else:
    print(
        negative_columns[
            ["Dataset", "Column", "Negative_Count",
             "Min_Value", "Max_Value"]
        ].to_string(index=False)
    )

COLUMNS WITH NEGATIVE VALUES
            Dataset        Column  Negative_Count  Min_Value  Max_Value
subscription_months mrr_delta_usd             626    -3547.9     3150.0


In [21]:
# Step 3.6 Part E.2 — MRR Consistency Validation

monthly_df = datasets["subscription_months"]

# Check 1: Negative MRR values
negative_mrr = int((monthly_df["mrr_usd"] < 0).sum())

negative_previous_mrr = int(
    (monthly_df["previous_mrr_usd"] < 0).sum()
)

# Check 2: Recalculate MRR difference
calculated_delta = (
    monthly_df["mrr_usd"]
    - monthly_df["previous_mrr_usd"]
)

# Allow tiny floating-point differences
delta_mismatch = int(
    (
        (calculated_delta - monthly_df["mrr_delta_usd"]).abs()
        > 0.01
    ).sum()
)

# Check 3: Missing MRR values
missing_mrr = int(
    monthly_df[
        ["mrr_usd", "previous_mrr_usd", "mrr_delta_usd"]
    ].isna().sum().sum()
)

print("MRR CONSISTENCY VALIDATION")
print("=" * 60)

print("Total monthly records:", len(monthly_df))
print("Negative current MRR:", negative_mrr)
print("Negative previous MRR:", negative_previous_mrr)
print("MRR delta mismatches:", delta_mismatch)
print("Missing MRR values:", missing_mrr)

if (
    negative_mrr == 0
    and negative_previous_mrr == 0
    and delta_mismatch == 0
    and missing_mrr == 0
):
    print("\nRESULT: PASS - MRR consistency checks passed.")
else:
    print("\nRESULT: REVIEW REQUIRED - MRR issues detected.")

MRR CONSISTENCY VALIDATION
Total monthly records: 19015
Negative current MRR: 0
Negative previous MRR: 0
MRR delta mismatches: 0
Missing MRR values: 0

RESULT: PASS - MRR consistency checks passed.


In [22]:
# Check MRR movement categories

print("MRR MOVEMENT DISTRIBUTION")
print("=" * 55)

print(
    monthly_df["movement_type"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nNegative MRR delta records:",
      int((monthly_df["mrr_delta_usd"] < 0).sum()))

print("Positive MRR delta records:",
      int((monthly_df["mrr_delta_usd"] > 0).sum()))

print("Zero MRR delta records:",
      int((monthly_df["mrr_delta_usd"] == 0).sum()))

MRR MOVEMENT DISTRIBUTION
movement_type
Retained        16143
New              1200
Expansion        1023
Churn             368
Contraction       258
Reactivation       23

Negative MRR delta records: 626
Positive MRR delta records: 2246
Zero MRR delta records: 16143


In [23]:
# Step 3.6 Part E.3 — Invoice Revenue Consistency

import numpy as np

invoice_df = datasets["invoices"]

# Check 1: Missing financial values
financial_columns = [
    "amount_usd",
    "discount_usd",
    "subtotal_usd",
    "tax_usd",
    "total_usd"
]

missing_financial = int(
    invoice_df[financial_columns].isna().sum().sum()
)

# Check 2: Negative financial values
negative_financial = int(
    (invoice_df[financial_columns] < 0).sum().sum()
)

# Check 3: Subtotal calculation
calculated_subtotal = (
    invoice_df["amount_usd"]
    - invoice_df["discount_usd"]
)

subtotal_mismatch = int(
    (
        ~np.isclose(
            calculated_subtotal,
            invoice_df["subtotal_usd"],
            atol=0.01,
            rtol=0
        )
    ).sum()
)

# Check 4: Final invoice total calculation
calculated_total = (
    invoice_df["subtotal_usd"]
    + invoice_df["tax_usd"]
)

total_mismatch = int(
    (
        ~np.isclose(
            calculated_total,
            invoice_df["total_usd"],
            atol=0.01,
            rtol=0
        )
    ).sum()
)

# Check 5: Discount cannot exceed original amount
excessive_discount = int(
    (
        invoice_df["discount_usd"]
        > invoice_df["amount_usd"]
    ).sum()
)

print("INVOICE REVENUE CONSISTENCY VALIDATION")
print("=" * 65)

print("Total invoice records:", len(invoice_df))
print("Missing financial values:", missing_financial)
print("Negative financial values:", negative_financial)
print("Subtotal calculation mismatches:", subtotal_mismatch)
print("Invoice total calculation mismatches:", total_mismatch)
print("Discounts greater than amount:", excessive_discount)

if (
    missing_financial == 0
    and negative_financial == 0
    and subtotal_mismatch == 0
    and total_mismatch == 0
    and excessive_discount == 0
):
    print("\nRESULT: PASS - Invoice revenue checks passed.")
else:
    print("\nRESULT: REVIEW REQUIRED - Invoice issues detected.")

INVOICE REVENUE CONSISTENCY VALIDATION
Total invoice records: 11257
Missing financial values: 0
Negative financial values: 0
Subtotal calculation mismatches: 0
Invoice total calculation mismatches: 0
Discounts greater than amount: 0

RESULT: PASS - Invoice revenue checks passed.


In [24]:
# Inspect invoice financial value ranges

print("INVOICE FINANCIAL SUMMARY")
print("=" * 70)

print(
    invoice_df[financial_columns]
    .agg(["min", "max", "mean"])
    .round(2)
    .to_string()
)

INVOICE FINANCIAL SUMMARY
      amount_usd  discount_usd  subtotal_usd   tax_usd  total_usd
min        39.00          0.00         35.10      0.00      35.10
max     83090.04      10918.02      83090.04  15787.11   98877.15
mean      746.11         24.82        721.29     69.39     790.68


In [25]:
# Step 3.6 Part F.1 — Cross-Table Consistency

subscriptions_df = datasets["subscriptions"]
monthly_df = datasets["subscription_months"]
invoices_df = datasets["invoices"]

# Subscription reference data
subscription_lookup = subscriptions_df[
    ["subscription_id", "customer_id", "plan_id"]
].copy()

# Validate monthly records against subscriptions
monthly_check = monthly_df[
    ["subscription_id", "customer_id", "plan_id"]
].merge(
    subscription_lookup,
    on="subscription_id",
    how="left",
    suffixes=("_monthly", "_subscription"),
    validate="many_to_one",
    indicator=True
)

monthly_missing_subscription = int(
    (monthly_check["_merge"] != "both").sum()
)

monthly_customer_mismatch = int(
    (
        (monthly_check["_merge"] == "both")
        & (
            monthly_check["customer_id_monthly"]
            != monthly_check["customer_id_subscription"]
        )
    ).sum()
)

monthly_plan_mismatch = int(
    (
        (monthly_check["_merge"] == "both")
        & (
            monthly_check["plan_id_monthly"]
            != monthly_check["plan_id_subscription"]
        )
    ).sum()
)

# Validate invoice customer against subscription customer
invoice_check = invoices_df[
    ["subscription_id", "customer_id"]
].merge(
    subscription_lookup[
        ["subscription_id", "customer_id"]
    ],
    on="subscription_id",
    how="left",
    suffixes=("_invoice", "_subscription"),
    validate="many_to_one",
    indicator=True
)

invoice_missing_subscription = int(
    (invoice_check["_merge"] != "both").sum()
)

invoice_customer_mismatch = int(
    (
        (invoice_check["_merge"] == "both")
        & (
            invoice_check["customer_id_invoice"]
            != invoice_check["customer_id_subscription"]
        )
    ).sum()
)

print("CROSS-TABLE CONSISTENCY VALIDATION")
print("=" * 65)

print("Monthly records checked:", len(monthly_check))
print("Monthly missing subscription references:",
      monthly_missing_subscription)
print("Monthly customer mismatches:",
      monthly_customer_mismatch)
print("Monthly plan mismatches:",
      monthly_plan_mismatch)

print("\nInvoice records checked:", len(invoice_check))
print("Invoice missing subscription references:",
      invoice_missing_subscription)
print("Invoice customer mismatches:",
      invoice_customer_mismatch)

CROSS-TABLE CONSISTENCY VALIDATION
Monthly records checked: 19015
Monthly missing subscription references: 0
Monthly customer mismatches: 0
Monthly plan mismatches: 0

Invoice records checked: 11257
Invoice missing subscription references: 0
Invoice customer mismatches: 0


In [26]:
# Overall cross-table consistency summary

total_relationship_issues = (
    monthly_missing_subscription
    + monthly_customer_mismatch
    + monthly_plan_mismatch
    + invoice_missing_subscription
    + invoice_customer_mismatch
)

print("Total cross-table discrepancies:",
      total_relationship_issues)

if total_relationship_issues == 0:
    print("RESULT: PASS - All checked relationships are consistent.")
else:
    print("RESULT: REVIEW REQUIRED - Investigate discrepancies.")

Total cross-table discrepancies: 0
RESULT: PASS - All checked relationships are consistent.


In [27]:
# Step 3.6 Part F.2 — Business Rule Validation

monthly_df = datasets["subscription_months"]

# Normalize is_active without changing the original DataFrame
active = (
    monthly_df["is_active"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False,
        "1": True,
        "0": False
    })
)

mrr = monthly_df["mrr_usd"]
delta = monthly_df["mrr_delta_usd"]
movement = monthly_df["movement_type"]

# Check invalid active-status values
invalid_active_status = int(active.isna().sum())

# Check active and inactive revenue consistency
active_zero_mrr = int(
    ((active == True) & (mrr <= 0)).sum()
)

inactive_positive_mrr = int(
    ((active == False) & (mrr > 0)).sum()
)

# Movement classification checks
churn_nonzero_mrr = int(
    ((movement == "Churn") & (mrr != 0)).sum()
)

expansion_invalid_delta = int(
    ((movement == "Expansion") & (delta <= 0)).sum()
)

contraction_invalid_delta = int(
    ((movement == "Contraction") & (delta >= 0)).sum()
)

retained_nonzero_delta = int(
    ((movement == "Retained") & (delta.abs() > 0.01)).sum()
)

business_rule_results = {
    "Invalid active-status values": invalid_active_status,
    "Active records with non-positive MRR": active_zero_mrr,
    "Inactive records with positive MRR": inactive_positive_mrr,
    "Churn records with nonzero MRR": churn_nonzero_mrr,
    "Expansion records with non-positive delta": expansion_invalid_delta,
    "Contraction records with non-negative delta": contraction_invalid_delta,
    "Retained records with nonzero delta": retained_nonzero_delta
}

print("SUBSCRIPTION BUSINESS RULE VALIDATION")
print("=" * 70)

for rule, count in business_rule_results.items():
    print(f"{rule}: {count}")

total_issues = sum(business_rule_results.values())

print("\nTotal rule violations:", total_issues)

if total_issues == 0:
    print("RESULT: PASS - Checked business rules are consistent.")
else:
    print("RESULT: REVIEW REQUIRED - Investigate rule violations.")

SUBSCRIPTION BUSINESS RULE VALIDATION
Invalid active-status values: 19015
Active records with non-positive MRR: 0
Inactive records with positive MRR: 0
Churn records with nonzero MRR: 0
Expansion records with non-positive delta: 0
Contraction records with non-negative delta: 0
Retained records with nonzero delta: 0

Total rule violations: 19015
RESULT: REVIEW REQUIRED - Investigate rule violations.


In [28]:
# Step 3.6 Part F.3 — Missing Values Business-Rule Audit

customers_df = datasets["customers"]
subscriptions_df = datasets["subscriptions"]
invoices_df = datasets["invoices"]

# Customers: missing churn dates by customer status
customer_missing = pd.crosstab(
    customers_df["customer_status"].fillna("MISSING_STATUS"),
    customers_df["churn_date"].isna(),
    rownames=["Customer_Status"],
    colnames=["Churn_Date_Missing"]
)

# Subscriptions: missing end dates by subscription status
subscription_missing = pd.crosstab(
    subscriptions_df["subscription_status"].fillna("MISSING_STATUS"),
    subscriptions_df["end_date"].isna(),
    rownames=["Subscription_Status"],
    colnames=["End_Date_Missing"]
)

# Invoices: missing payment duration by invoice status
invoice_missing = pd.crosstab(
    invoices_df["invoice_status"].fillna("MISSING_STATUS"),
    invoices_df["days_to_payment"].isna(),
    rownames=["Invoice_Status"],
    colnames=["Days_To_Payment_Missing"]
)

print("CUSTOMERS — CHURN DATE MISSINGNESS")
print(customer_missing.to_string())

print("\nSUBSCRIPTIONS — END DATE MISSINGNESS")
print(subscription_missing.to_string())

print("\nINVOICES — PAYMENT DURATION MISSINGNESS")
print(invoice_missing.to_string())

CUSTOMERS — CHURN DATE MISSINGNESS
Churn_Date_Missing  False  True 
Customer_Status                 
Active                  0    855
Churned               345      0

SUBSCRIPTIONS — END DATE MISSINGNESS
End_Date_Missing     False  True 
Subscription_Status              
Active                   0    855
Churned                368      0
Downgraded              43      0
Upgraded               115      0

INVOICES — PAYMENT DURATION MISSINGNESS
Days_To_Payment_Missing  False  True 
Invoice_Status                       
Failed                       0    309
Overdue                    525      0
Paid                     10316      0
Refunded                   107      0


In [29]:
# Verify the three known missing-value totals

print("MISSING VALUE VERIFICATION")
print("=" * 55)

print(
    "Customers missing churn_date:",
    int(customers_df["churn_date"].isna().sum())
)

print(
    "Subscriptions missing end_date:",
    int(subscriptions_df["end_date"].isna().sum())
)

print(
    "Invoices missing days_to_payment:",
    int(invoices_df["days_to_payment"].isna().sum())
)

total_missing = (
    customers_df["churn_date"].isna().sum()
    + subscriptions_df["end_date"].isna().sum()
    + invoices_df["days_to_payment"].isna().sum()
)

print("\nTotal missing cells:", int(total_missing))

MISSING VALUE VERIFICATION
Customers missing churn_date: 855
Subscriptions missing end_date: 855
Invoices missing days_to_payment: 309

Total missing cells: 2019


In [30]:
print("1. CUSTOMER STATUS vs MISSING CHURN DATE")
print(customer_missing.to_string())

print("\n2. SUBSCRIPTION STATUS vs MISSING END DATE")
print(subscription_missing.to_string())

print("\n3. INVOICE STATUS vs MISSING PAYMENT DAYS")
print(invoice_missing.to_string())

1. CUSTOMER STATUS vs MISSING CHURN DATE
Churn_Date_Missing  False  True 
Customer_Status                 
Active                  0    855
Churned               345      0

2. SUBSCRIPTION STATUS vs MISSING END DATE
End_Date_Missing     False  True 
Subscription_Status              
Active                   0    855
Churned                368      0
Downgraded              43      0
Upgraded               115      0

3. INVOICE STATUS vs MISSING PAYMENT DAYS
Days_To_Payment_Missing  False  True 
Invoice_Status                       
Failed                       0    309
Overdue                    525      0
Paid                     10316      0
Refunded                   107      0


### Step 3.6 Part F.3 — Missing Values Business-Rule Audit

**Result: PASS**

- Total missing cells: 2,019
- Customers: 855 missing churn dates, all associated with Active customers.
- Subscriptions: 855 missing end dates, all associated with Active subscriptions.
- Invoices: 309 missing days_to_payment values, all associated with Failed invoices.
- Unexpected missing-value combinations: 0

**Decision:** Retain missing values as they are consistent with
the observed business statuses. Do not apply automatic
imputation or row deletion.

**Note:** Payment-duration interpretation for Overdue invoices
should follow the dataset's field definition.